In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/corpus/positive_pairs.json


# Phase 0A — CSN Judge Calibration (RQ3)


*Compares Llama-8B/ Qwen-3.5B / Qwen-coder-7B against CodeSearchNet human judgements.Metrics: Quadratic Weighted Kappa (QWK) + Kendall-τ*


In [2]:
!pip install -q python-terrier pyterrier-dr datasets ir_datasets scipy scikit-learn accelerate groq

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.8/208.8 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 25.7 MB/s eta 0:00:00


In [3]:
import os, json, gc, re, requests
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
from scipy.stats import kendalltau
from sklearn.metrics import cohen_kappa_score
import pyterrier as pt
import pyterrier_dr
pd.set_option('display.max_colwidth', 200)
print('PyTerrier:', pt.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

2026-04-01 15:32:41.920205: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775057562.135473      54 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775057562.199559      54 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775057562.712138      54 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775057562.712181      54 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775057562.712184      54 computation_placer.cc:177] computation placer alr

PyTerrier: 1.0.4
CUDA: True
GPU: Tesla T4


#### only taking the python codes from CSN to evalute

In [4]:
import ir_datasets
import pandas as pd

# 1. Load the dataset
dataset = ir_datasets.load("codesearchnet/challenge")

# 2. Extract Queries into a DataFrame
queries_df = pd.DataFrame(
    [{"query_id": q.query_id, "query_text": q.text} for q in dataset.queries_iter()]
)

# 3. Extract Qrels into a DataFrame
qrels_df = pd.DataFrame(
    [{"query_id": q.query_id, "doc_id": q.doc_id, "relevance": q.relevance} for q in dataset.qrels_iter()]
)

# 4. Extract Documents into a DataFrame (Filtering for Python ONLY)
python_docs = []
for doc in dataset.docs_iter():
    if doc.language == 'python':
        python_docs.append({
            "doc_id": doc.doc_id,
            "code": doc.code,               # The actual python code
            "repo": doc.repo,               # The github repository name (like 'huggingface/transformers')
            "path": doc.path,               # The file path in the repository
            "func_name": doc.func_name      # The exact function name 
        })
        
docs_df = pd.DataFrame(python_docs)

# 5. Merge everything together
merged_df = pd.merge(qrels_df, queries_df, on="query_id", how="inner")
documents_csn= pd.merge(merged_df, docs_df, on="doc_id", how="inner")

print(f"Total Python Relevance Judgements found: {len(documents_csn)}")


[INFO] [starting] https://raw.githubusercontent.com/github/CodeSearchNet/master/resources/queries.csv
[INFO] [finished] https://raw.githubusercontent.com/github/CodeSearchNet/master/resources/queries.csv: [00:00] [2.49kB] [4.43MB/s]
[INFO] [starting] https://raw.githubusercontent.com/github/CodeSearchNet/master/resources/annotationStore.csv   
[INFO] [finished] https://raw.githubusercontent.com/github/CodeSearchNet/master/resources/annotationStore.csv: [00:00] [678kB] [25.5MB/s]
[INFO] If you have a local copy of https://huggingface.co/datasets/macavaney/codesearchnet-mirror/resolve/main/v2/python.zip, you can symlink it here to avoid downloading it again: /root/.ir_datasets/downloads/07b49dd01fbac894fbdae22da6462e4f
[INFO] [starting] https://huggingface.co/datasets/macavaney/codesearchnet-mirror/resolve/main/v2/python.zip
[INFO] [finished] https://huggingface.co/datasets/macavaney/codesearchnet-mirror/resolve/main/v2/python.zip: [00:04] [941MB] [203MB/s]
[INFO] If you have a local cop

Total Python Relevance Judgements found: 1109


In [5]:
# Basic check
display(documents_csn.head())

,query_id,doc_id,relevance,query_text,code,repo,path,func_name
0,64,https://github.com/JoseAntFer/pyny3d/blob/fb81684935a24f7e50c975cb4383c81a63ab56df/pyny3d/utils.py#L8-L33,2,sorting multiple arrays based on another arrays sorted order,"def sort_numpy(array, col=0, order_back=False):\r\n """"""\r\n Sorts the columns for an entire ``ndarrray`` according to sorting\r\n one of them.\r\n \r\n :param array: Array to sort.\...",JoseAntFer/pyny3d,pyny3d/utils.py,sort_numpy
1,2,https://github.com/keon/algorithms/blob/4d6569464a62a75c1357acc97e2dd32ee2f9f4a3/algorithms/queues/priority_queue.py#L38-L49,3,priority queue,"def push(self, item, priority=None):\n """"""Push the item in the priority queue.\n if priority is not given, priority is set to the value of item.\n """"""\n priority = item...",keon/algorithms,algorithms/queues/priority_queue.py,PriorityQueue.push
2,60,https://github.com/mapbox/mapbox-sdk-py/blob/72d19dbcf2d254a6ea08129a726471fd21f13023/mapbox/services/base.py#L122-L144,3,custom http error response,"def handle_http_error(self, response, custom_messages=None,\n raise_for_status=False):\n """"""Converts service errors to Python exceptions\n\n Parameters\n ...",mapbox/mapbox-sdk-py,mapbox/services/base.py,Service.handle_http_error
3,61,https://github.com/a-tal/nagaram/blob/2edcb0ef8cb569ebd1c398be826472b4831d6110/nagaram/scrabble.py#L136-L184,0,how to determine a string is a valid word,"def valid_scrabble_word(word):\n """"""Checks if the input word could be played with a full bag of tiles.\n\n Returns:\n True or false\n """"""\n\n letters_in_bag = {\n ""a"": 9,...",a-tal/nagaram,nagaram/scrabble.py,valid_scrabble_word
4,80,https://github.com/rkargon/pixelsorter/blob/0775d1e487fbcb023e411e1818ba3290b0e8665e/pixelsorter/util.py#L72-L84,2,randomly extract x items from a list,"def weighted_random_choice(items):\n """"""\n Returns a weighted random choice from a list of items.\n :param items: A list of tuples (object, weight)\n :return: A random object, whose li...",rkargon/pixelsorter,pixelsorter/util.py,weighted_random_choice


In [6]:
documents_csn.to_csv('/kaggle/working/documents_csn.tsv',sep='\t',index=False)

In [7]:
documents_csn=pd.read_csv('/kaggle/working/documents_csn.tsv',sep='\t',index_col=0)

In [8]:
documents_csn

,doc_id,relevance,query_text,code,repo,path,func_name
query_id,,,,,,,
64,https://github.com/JoseAntFer/pyny3d/blob/fb81684935a24f7e50c975cb4383c81a63ab56df/pyny3d/utils.py#L8-L33,2,sorting multiple arrays based on another arrays sorted order,"def sort_numpy(array, col=0, order_back=False):\r\n """"""\r\n Sorts the columns for an entire ``ndarrray`` according to sorting\r\n one of them.\r\n \r\n :param array: Array to sort.\...",JoseAntFer/pyny3d,pyny3d/utils.py,sort_numpy
2,https://github.com/keon/algorithms/blob/4d6569464a62a75c1357acc97e2dd32ee2f9f4a3/algorithms/queues/priority_queue.py#L38-L49,3,priority queue,"def push(self, item, priority=None):\n """"""Push the item in the priority queue.\n if priority is not given, priority is set to the value of item.\n """"""\n priority = item...",keon/algorithms,algorithms/queues/priority_queue.py,PriorityQueue.push
60,https://github.com/mapbox/mapbox-sdk-py/blob/72d19dbcf2d254a6ea08129a726471fd21f13023/mapbox/services/base.py#L122-L144,3,custom http error response,"def handle_http_error(self, response, custom_messages=None,\n raise_for_status=False):\n """"""Converts service errors to Python exceptions\n\n Parameters\n ...",mapbox/mapbox-sdk-py,mapbox/services/base.py,Service.handle_http_error
61,https://github.com/a-tal/nagaram/blob/2edcb0ef8cb569ebd1c398be826472b4831d6110/nagaram/scrabble.py#L136-L184,0,how to determine a string is a valid word,"def valid_scrabble_word(word):\n """"""Checks if the input word could be played with a full bag of tiles.\n\n Returns:\n True or false\n """"""\n\n letters_in_bag = {\n ""a"": 9,...",a-tal/nagaram,nagaram/scrabble.py,valid_scrabble_word
80,https://github.com/rkargon/pixelsorter/blob/0775d1e487fbcb023e411e1818ba3290b0e8665e/pixelsorter/util.py#L72-L84,2,randomly extract x items from a list,"def weighted_random_choice(items):\n """"""\n Returns a weighted random choice from a list of items.\n :param items: A list of tuples (object, weight)\n :return: A random object, whose li...",rkargon/pixelsorter,pixelsorter/util.py,weighted_random_choice
...,...,...,...,...,...,...,...
55,https://github.com/simonw/datasette/blob/11b352b4d52fd02a422776edebb14f12e4994d3b/datasette/app.py#L521-L527,1,how to get database table name,"def table_metadata(self, database, table):\n ""Fetch table-specific metadata.""\n return (self.metadata(""databases"") or {}).get(database, {}).get(\n ""tables"", {}\n )....",simonw/datasette,datasette/app.py,Datasette.table_metadata
39,https://github.com/apple/turicreate/blob/74514c3f99e25b46f22c6e02977fe3da69221c2e/src/external/coremltools_wrap/coremltools/deps/protobuf/python/google/protobuf/descriptor.py#L321-L337,3,get name of enumerated value,"def EnumValueName(self, enum, value):\n """"""Returns the string name of an enum value.\n\n This is just a small helper method to simplify a common operation.\n\n Args:\n enum: string n...",apple/turicreate,src/external/coremltools_wrap/coremltools/deps/protobuf/python/google/protobuf/descriptor.py,Descriptor.EnumValueName
47,https://github.com/nion-software/nionswift/blob/d43693eaf057b8683b9638e575000f055fede452/nion/swift/model/NDataHandler.py#L469-L481,1,read properties file,"def read_properties(self):\n """"""\n Read properties from the ndata file reference\n\n :param reference: the reference from which to read\n :return: a tuple o...",nion-software/nionswift,nion/swift/model/NDataHandler.py,NDataHandler.read_properties


In [9]:
# 1. Move query_id from the Index back into a regular column
documents_csn = documents_csn.reset_index()
# 2. Now your rename will work perfectly
documents_csn = documents_csn.rename(columns={
    'query_id': 'qid',
    'doc_id': 'docno',
    'query_text': 'query',
    'relevance':  'human_relevance',
})

In [10]:
documents_csn['qid']   = documents_csn['qid'].astype(str)
documents_csn['docno'] = documents_csn['docno'].astype(str)

In [11]:
documents_csn = documents_csn.dropna(subset=['code'])
print(f'Total human-judged pairs: {len(documents_csn)}')
print('Relevance distribution:')
print(documents_csn['human_relevance'].value_counts().sort_index())

Total human-judged pairs: 1109
Relevance distribution:
human_relevance
0    244
1    291
2    274
3    300
Name: count, dtype: int64


In [12]:
documents_csn

,qid,docno,human_relevance,query,code,repo,path,func_name
0,64,https://github.com/JoseAntFer/pyny3d/blob/fb81684935a24f7e50c975cb4383c81a63ab56df/pyny3d/utils.py#L8-L33,2,sorting multiple arrays based on another arrays sorted order,"def sort_numpy(array, col=0, order_back=False):\r\n """"""\r\n Sorts the columns for an entire ``ndarrray`` according to sorting\r\n one of them.\r\n \r\n :param array: Array to sort.\...",JoseAntFer/pyny3d,pyny3d/utils.py,sort_numpy
1,2,https://github.com/keon/algorithms/blob/4d6569464a62a75c1357acc97e2dd32ee2f9f4a3/algorithms/queues/priority_queue.py#L38-L49,3,priority queue,"def push(self, item, priority=None):\n """"""Push the item in the priority queue.\n if priority is not given, priority is set to the value of item.\n """"""\n priority = item...",keon/algorithms,algorithms/queues/priority_queue.py,PriorityQueue.push
2,60,https://github.com/mapbox/mapbox-sdk-py/blob/72d19dbcf2d254a6ea08129a726471fd21f13023/mapbox/services/base.py#L122-L144,3,custom http error response,"def handle_http_error(self, response, custom_messages=None,\n raise_for_status=False):\n """"""Converts service errors to Python exceptions\n\n Parameters\n ...",mapbox/mapbox-sdk-py,mapbox/services/base.py,Service.handle_http_error
3,61,https://github.com/a-tal/nagaram/blob/2edcb0ef8cb569ebd1c398be826472b4831d6110/nagaram/scrabble.py#L136-L184,0,how to determine a string is a valid word,"def valid_scrabble_word(word):\n """"""Checks if the input word could be played with a full bag of tiles.\n\n Returns:\n True or false\n """"""\n\n letters_in_bag = {\n ""a"": 9,...",a-tal/nagaram,nagaram/scrabble.py,valid_scrabble_word
4,80,https://github.com/rkargon/pixelsorter/blob/0775d1e487fbcb023e411e1818ba3290b0e8665e/pixelsorter/util.py#L72-L84,2,randomly extract x items from a list,"def weighted_random_choice(items):\n """"""\n Returns a weighted random choice from a list of items.\n :param items: A list of tuples (object, weight)\n :return: A random object, whose li...",rkargon/pixelsorter,pixelsorter/util.py,weighted_random_choice
...,...,...,...,...,...,...,...,...
1104,55,https://github.com/simonw/datasette/blob/11b352b4d52fd02a422776edebb14f12e4994d3b/datasette/app.py#L521-L527,1,how to get database table name,"def table_metadata(self, database, table):\n ""Fetch table-specific metadata.""\n return (self.metadata(""databases"") or {}).get(database, {}).get(\n ""tables"", {}\n )....",simonw/datasette,datasette/app.py,Datasette.table_metadata
1105,39,https://github.com/apple/turicreate/blob/74514c3f99e25b46f22c6e02977fe3da69221c2e/src/external/coremltools_wrap/coremltools/deps/protobuf/python/google/protobuf/descriptor.py#L321-L337,3,get name of enumerated value,"def EnumValueName(self, enum, value):\n """"""Returns the string name of an enum value.\n\n This is just a small helper method to simplify a common operation.\n\n Args:\n enum: string n...",apple/turicreate,src/external/coremltools_wrap/coremltools/deps/protobuf/python/google/protobuf/descriptor.py,Descriptor.EnumValueName
1106,47,https://github.com/nion-software/nionswift/blob/d43693eaf057b8683b9638e575000f055fede452/nion/swift/model/NDataHandler.py#L469-L481,1,read properties file,"def read_properties(self):\n """"""\n Read properties from the ndata file reference\n\n :param reference: the reference from which to read\n :return: a tuple o...",nion-software/nionswift,nion/swift/model/NDataHandler.py,NDataHandler.read_properties
1107,8,https://github.com/materialsproject/pymatgen/blob/4ca558cf72f8d5f8a1f21dfdfc0181a971c186da/pymatgen/io/abinit/flows.py#L254-L266,2,set working directory,"def set_workdir(self, workdir, chroot=False):\n """"""\n Set the working directory. Cannot be set more than once unless chroot is True\n """"""\n if not chroot and hasattr(se...",materialsproject/pymatgen,pymatgen/io/abinit/flows.py,Flow.set_workdir


In [13]:
# No sampling needed — use everything
calibration_df = documents_csn.copy()

calibration_df.to_csv('/kaggle/working/calibration_pool.csv', index=False)
print(f'Calibration pool: {len(calibration_df)} pairs')
print(calibration_df['human_relevance'].value_counts().sort_index())

Calibration pool: 1109 pairs
human_relevance
0    244
1    291
2    274
3    300
Name: count, dtype: int64


In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

RELEVANCE_PROMPT = '''\
You are an expert code relevance assessor for information retrieval research.

Task: Rate the relevance of a code snippet to a natural language query on a scale of 0-3.

Relevance Scale:
- 0: Not relevant - The code has no relation to the query
- 1: Marginally relevant - The code is tangentially related but does not answer the query
- 2: Relevant - The code addresses the query or provides related functionality
- 3: Highly relevant - The code directly and completely answers the query

Query: {query}

Code Snippet:
{code}

Instructions:
1. Analyze the semantic relationship between the query and code
2. Consider functionality, intent, and implementation details
3. Provide ONLY a single digit (0, 1, 2, or 3) as your response

Relevance Score:'''


## Utility functions

In [32]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
import time
import os, re, time

def load_llm(model_name):
    print(f'Loading {model_name}...')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.padding_side = 'left'
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map='auto',
    )
    print('  Loaded.')
    return tokenizer, model



def judge_batch(queries, codes, tokenizer, model, batch_size=8):
    '''Returns list of int scores 0-3 for each (query, code) pair.'''
    results = []
    batches = range(0, len(queries), batch_size)
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    for i in tqdm(batches, desc="Judging", unit="batch"):
        bq = queries[i:i+batch_size]
        bc = codes[i:i+batch_size]
        prompts = [RELEVANCE_PROMPT.format(query=q, code=c[:1000])
                   for q, c in zip(bq, bc)]
        inputs = tokenizer(
            prompts, return_tensors='pt', padding=True,
            truncation=True, max_length=2048, return_attention_mask=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            out = model.generate(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_new_tokens=5, do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        for j, o in enumerate(out):
            gen = tokenizer.decode(
                o[inputs['input_ids'][j].shape[0]:],
                skip_special_tokens=True
            ).strip()
            try:
                score = int(gen[0])
                if score not in [0, 1, 2, 3]: score = 0
            except:
                score = 0
            results.append(score)
    return results


In [16]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

HF_TOKEN=user_secrets.get_secret("HF_TOKEN")

In [17]:
from huggingface_hub import login
import os


login(token=HF_TOKEN)

In [36]:
# ── Judge with Llama-3.1-8B-Instruct ──
tok8b, mdl8b = load_llm('meta-llama/Llama-3.1-8B-Instruct')
calibration_df = pd.read_csv('/kaggle/working/calibration_pool.csv')
calibration_df['llama8b'] = judge_batch(
    calibration_df['query'].tolist(),
    calibration_df['code'].tolist(),
    tok8b, mdl8b, batch_size=16
)
calibration_df.to_csv('/kaggle/working/calibration_llama8b.csv', index=False)
print('Llama-8B done.')
print(calibration_df['llama8b'].value_counts().sort_index())

#=---- cleanup memory
mdl8b= None
tok8b= None
gc.collect()
for i in range(torch.cuda.device_count()):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
for i in range(torch.cuda.device_count()):
    print(f"GPU {i} → {torch.cuda.memory_allocated(i)/1e9:.2f} GB")

Loading meta-llama/Llama-3.1-8B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

  Loaded.


Judging:   0%|          | 0/70 [00:00<?, ?batch/s]

Llama-8B done.
llama8b
0    235
1     21
2    379
3    474
Name: count, dtype: int64
GPU 0 → 0.01 GB
GPU 1 → 0.01 GB


In [39]:
# ── Judge with Qwen2.5-3B-Instruct ──
tok_qwen, mdl_qwen = load_llm('Qwen/Qwen2.5-3B-Instruct')
calibration_df = pd.read_csv('/kaggle/working/calibration_llama8b.csv')
calibration_df['qwen3b'] = judge_batch(
    calibration_df['query'].tolist(),
    calibration_df['code'].tolist(),
    tok_qwen, mdl_qwen, batch_size=16
)
calibration_df.to_csv('/kaggle/working/calibration_qwen3b.csv', index=False)
print('Qwen-3B done.')
print(calibration_df['qwen3b'].value_counts().sort_index())

## clear loaded model
mdl_qwen= None
tok_qwen = None
gc.collect()
for i in range(torch.cuda.device_count()):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
for i in range(torch.cuda.device_count()):
    print(f"GPU {i} → {torch.cuda.memory_allocated(i)/1e9:.2f} GB")

Loading Qwen/Qwen2.5-3B-Instruct...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  Loaded.


Judging:   0%|          | 0/70 [00:00<?, ?batch/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Qwen-3B done.
qwen3b
0    122
1    352
2    625
3     10
Name: count, dtype: int64
GPU 0 → 0.01 GB
GPU 1 → 0.01 GB


In [40]:
tok_coder, mdl_coder = load_llm('Qwen/Qwen2.5-Coder-7B-Instruct')
calibration_df = pd.read_csv('/kaggle/working/calibration_qwen3b.csv')
calibration_df['qwen_coder7b'] = judge_batch(
    calibration_df['query'].tolist(),
    calibration_df['code'].tolist(),
    tok_coder, mdl_coder,
    batch_size=16    # fits easily on 2×T4
)
calibration_df.to_csv('/kaggle/working/calibration_all_models.csv', index=False)
print(calibration_df['qwen_coder7b'].value_counts().sort_index())
## clear loaded model
mdl_coder = None
tok_coder = None
gc.collect()
for i in range(torch.cuda.device_count()):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
for i in range(torch.cuda.device_count()):
    print(f"GPU {i} → {torch.cuda.memory_allocated(i)/1e9:.2f} GB")

Loading Qwen/Qwen2.5-Coder-7B-Instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  Loaded.


Judging:   0%|          | 0/70 [00:00<?, ?batch/s]

qwen_coder7b done.
qwen_coder7b
0    608
1      4
2    327
3    170
Name: count, dtype: int64
GPU 0 → 0.01 GB
GPU 1 → 0.01 GB


In [41]:
# CSN Calibration Metrics: Quadratic Weighted Kappa + Kendall-tau
calibration_df = pd.read_csv('/kaggle/working/calibration_all_models.csv')
human_scores   = calibration_df['human_relevance'].tolist()

model_cols = ['llama8b', 'qwen3b', 'qwen_coder7b']
rows = []
for m in model_cols:
    llm_scores = calibration_df[m].tolist()
    qwk = cohen_kappa_score(human_scores, llm_scores, weights='quadratic')
    kt  = kendalltau(human_scores, llm_scores).statistic
    rows.append({'model': m, 'QWK': round(qwk, 4), 'Kendall_tau': round(kt, 4)})

calibration_result = pd.DataFrame(rows).sort_values('QWK', ascending=False)
print('=== CSN Calibration Results ===')
display(calibration_result)

=== CSN Calibration Results ===


,model,QWK,Kendall_tau
2,qwen_coder7b,0.4369,0.4254
0,llama8b,0.4008,0.3957
1,qwen3b,0.3275,0.3312


# Phase 0B — CoSQA Binary Calibration
Binary human labels (0/1) vs LLM scores (0–3).  
**Cohen's κ** (binary): binarize LLM ≥ 2 → 1  
**Kendall-τ**: rank correlation between raw LLM 0-3 and binary 0/1 label (valid — τ is rank-based)

In [42]:
# Download CoSQA (200 pairs: 100 positive + 100 negative)
cosqa_url ='https://raw.githubusercontent.com/Jun-jie-Huang/CoCLR/refs/heads/main/data/qa/cosqa-all.json'
resp = requests.get(cosqa_url, timeout=60)
cosqa_raw = resp.json()
cosqa_full = pd.DataFrame(cosqa_raw).rename(columns={'doc': 'query', 'label': 'cosqa_label'})
print(f'CoSQA total: {len(cosqa_full)}')
print(cosqa_full['cosqa_label'].value_counts())



CoSQA total: 20604
cosqa_label
0    10584
1    10020
Name: count, dtype: int64


In [43]:
cosqa_full

,idx,query,code,code_tokens,docstring_tokens,cosqa_label
0,cosqa-train-18103,python remove all empty items in list,"def remove_empty_text(utterances: List[Utterance]) -> List[Utterance]:\n """"""Remove empty utterances from a list of utterances\n Args:\n utterances: The list of utterance we are proces...","def remove_empty_text ( utterances : List [ Utterance ] ) -> List [ Utterance ] : return [ utter for utter in utterances if utter . text . strip ( ) != """" ]",Remove empty utterances from a list of utterances Args : utterances : The list of utterance we are processing,0
1,cosqa-train-11841,python return property objectno not value,"def value(self):\n """"""Value of property.""""""\n if self._prop.fget is None:\n raise AttributeError('Unable to read attribute')\n return self._prop.fget(self._obj)",def value ( self ) : if self . _prop . fget is None : raise AttributeError ( 'Unable to read attribute' ) return self . _prop . fget ( self . _obj ),Value of property .,1
2,cosqa-train-3015,python not null dict,"def purge_dict(idict):\n """"""Remove null items from a dictionary """"""\n odict = {}\n for key, val in idict.items():\n if is_null(val):\n continue\n odict[key] = val...","def purge_dict ( idict ) : odict = { } for key , val in idict . items ( ) : if is_null ( val ) : continue odict [ key ] = val return odict",Remove null items from a dictionary,0
3,cosqa-train-5561,how to turn of traceback in python,"def format_exception(e):\n """"""Returns a string containing the type and text of the exception.\n\n """"""\n from .utils.printing import fill\n return '\n'.join(fill(line) for line in trace...","def format_exception ( e ) : from . utils . printing import fill return '\n' . join ( fill ( line ) for line in traceback . format_exception_only ( type ( e ) , e ) )",Returns a string containing the type and text of the exception .,0
4,cosqa-train-9028,generate list of fixed size python,"def batch(items, size):\n """"""Batches a list into a list of lists, with sub-lists sized by a specified\n batch size.""""""\n return [items[x:x + size] for x in xrange(0, len(items), size)]","def batch ( items , size ) : return [ items [ x : x + size ] for x in xrange ( 0 , len ( items ) , size ) ]",Batches a list into a list of lists with sub - lists sized by a specified batch size .,1
...,...,...,...,...,...,...
20599,cosqa-train-10131,multiple text files word count for each file python,"def count_words(file):\n """""" Counts the word frequences in a list of sentences.\n\n Note:\n This is a helper function for parallel execution of `Vocabulary.from_text`\n method.\n """"""\n c...","def count_words ( file ) : c = Counter ( ) with open ( file , 'r' ) as f : for l in f : words = l . strip ( ) . split ( ) c . update ( words ) return c",Counts the word frequences in a list of sentences .,1
20600,cosqa-train-19367,how to check a file is empty in python,"def _cnx_is_empty(in_file):\n """"""Check if cnr or cns files are empty (only have a header)\n """"""\n with open(in_file) as in_handle:\n for i, line in enumerate(in_handle):\n ...","def _cnx_is_empty ( in_file ) : with open ( in_file ) as in_handle : for i , line in enumerate ( in_handle ) : if i > 0 : return False return True",Check if cnr or cns files are empty ( only have a header ),1
20601,cosqa-dev-344,python rpy2 robjects not found,"def setup_detect_python2():\n """"""\n Call this before using the refactoring tools to create them on demand\n if needed.\n """"""\n if None in [RTs._rt_py2_detect, RT...","def setup_detect_python2 ( ) : if None in [ RTs . _rt_py2_detect , RTs . _rtp_py2_detect ] : RTs . _rt_py2_detect = RefactoringTool ( py2_detect_fixers ) RTs . _rtp_py2_detect = RefactoringTool ( ...",Call this before using the refactoring tools to create them on demand if needed .,0
20602,cosqa-train-4318,ask if a method can see a variable python,"def is_parameter(self):\n """"""Whether this is a function parameter.""""""\n return (isinstance(self.scope, CodeFunction)\n and self in self.scope.pa

In [44]:
cosqa_sample = pd.concat([
    cosqa_full[cosqa_full['cosqa_label'] == 1].sample(100, random_state=42),
    cosqa_full[cosqa_full['cosqa_label'] == 0].sample(100, random_state=42),
]).reset_index(drop=True)
print(f'\nSampled {len(cosqa_sample)} pairs (100 pos + 100 neg)')
cosqa_sample.to_csv('/kaggle/working/cosqa_sample.csv', index=False)


Sampled 200 pairs (100 pos + 100 neg)


In [46]:
# Judge CoSQA with all 3 LLMs (fast — only 200 pairs each)
cosqa_sample = pd.read_csv('/kaggle/working/cosqa_sample.csv')

for model_name, hf_name, use_4bit, col in [
    ('llama8b',  'meta-llama/Llama-3.1-8B-Instruct',  False, 'llama8b_raw'),
    ('qwen3b',   'Qwen/Qwen2.5-3B-Instruct',          False, 'qwen3b_raw'),
    ('qwen_coder7b', 'Qwen/Qwen2.5-Coder-7B-Instruct', True,  'qwen_coder7b_raw'),
]:
    
    tok, mdl = load_llm(hf_name)
    cosqa_sample[col] = judge_batch(
        cosqa_sample['query'].tolist(),
        cosqa_sample['code'].tolist(),
        tok, mdl,
        batch_size=16
    )
    ## unload llm-----
    mdl = None
    tok = None
    gc.collect()
    for i in range(torch.cuda.device_count()):
        with torch.cuda.device(i):
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i} → {torch.cuda.memory_allocated(i)/1e9:.2f} GB")
    cosqa_sample.to_csv('/kaggle/working/cosqa_judged.csv', index=False)
    print(cosqa_sample[col].value_counts().sort_index())


Loading meta-llama/Llama-3.1-8B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

  Loaded.


Judging:   0%|          | 0/13 [00:00<?, ?batch/s]

GPU 0 → 0.01 GB
GPU 1 → 0.01 GB
llama8b_raw
0    20
1     4
2    77
3    99
Name: count, dtype: int64
Loading Qwen/Qwen2.5-3B-Instruct...


...

GPU 0 → 0.01 GB
GPU 1 → 0.01 GB
qwen3b_raw
0     10
1     47
2    140
3      3
Name: count, dtype: int64
Loading Qwen/Qwen2.5-Coder-7B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

  Loaded.


Judging:   0%|          | 0/13 [00:00<?, ?batch/s]

GPU 0 → 0.01 GB
GPU 1 → 0.01 GB
qwen_coder7b_raw
0    99
1     2
2    72
3    27
Name: count, dtype: int64


In [47]:
# CoSQA Metrics: binary Cohen's κ + Kendall-τ (ordinal vs binary)
cosqa_sample = pd.read_csv('/kaggle/working/cosqa_judged.csv')
human_bin    = cosqa_sample['cosqa_label'].tolist()

def binarize(scores, threshold=2):
    # Justification: 0-3 scale; 2='Relevant' maps to positive (1)
    return [1 if s >= threshold else 0 for s in scores]

cosqa_rows = []
for col, label in [('llama8b_raw','llama8b'),('qwen3b_raw','qwen3b'),('qwen_coder7b_raw','qwen_coder7b')]:
    raw   = cosqa_sample[col].tolist()
    llm_b = binarize(raw)
    kappa = cohen_kappa_score(human_bin, llm_b)
    # Kendall-tau: rank correlation between ordinal LLM scores and binary human labels
    kt    = kendalltau(human_bin, raw).statistic
    cosqa_rows.append({
        'model': label,
        'Binary_Cohen_Kappa': round(kappa, 4),
        'Kendall_tau_ordinal': round(kt, 4)
    })

cosqa_result = pd.DataFrame(cosqa_rows)
print('=== CoSQA Calibration Results ===')
display(cosqa_result)

=== CoSQA Calibration Results ===


,model,Binary_Cohen_Kappa,Kendall_tau_ordinal
0,llama8b,-0.02,-0.0016
1,qwen3b,0.03,0.0280
2,qwen_coder7b,-0.01,-0.0034


In [48]:
# ── Master Judge Selection Table ──
summary = calibration_result.merge(cosqa_result, on='model', how='left')
summary['mean_score'] = summary[['QWK','Kendall_tau','Binary_Cohen_Kappa','Kendall_tau_ordinal']].mean(axis=1)
summary = summary.sort_values('mean_score', ascending=False).reset_index(drop=True)

print('=== FULL CALIBRATION SUMMARY (higher = better judge) ===')
display(summary)

# MODEL_MAP = {
#     'llama8b':  'meta-llama/Llama-3.1-8B-Instruct',
#     'qwen3b':   'Qwen/Qwen2.5-3B-Instruct',
#     'qwen_coder7b': 'Qwen/Qwen2.5-Coder-7B-Instruct',
# }
# BEST_JUDGE       = summary.iloc[0]['model']
# BEST_JUDGE_HF    = MODEL_MAP[BEST_JUDGE]
# BEST_JUDGE_4BIT  = (BEST_JUDGE == 'llama70b')
# print(f'\nBEST JUDGE: {BEST_JUDGE}  →  {BEST_JUDGE_HF}')
# summary.to_csv('/kaggle/working/judge_selection.csv', index=False)

=== FULL CALIBRATION SUMMARY (higher = better judge) ===


,model,QWK,Kendall_tau,Binary_Cohen_Kappa,Kendall_tau_ordinal,mean_score
0,qwen_coder7b,0.4369,0.4254,-0.01,-0.0034,0.212225
1,llama8b,0.4008,0.3957,-0.02,-0.0016,0.193725
2,qwen3b,0.3275,0.3312,0.03,0.0280,0.179175


# Phase 1 — Custom GenAI Corpus (5 repos)
Load `positive_pairs.json`, build corpus (~1000 docs) and 21 topics.

In [49]:
import numpy as np
import pandas as pd

df=pd.read_json('/kaggle/input/corpus/positive_pairs.json')
df.head(10)

,document_id,anchor,positive,framework,source
0,doc_c73087bba37c14c1e003ab940014b8a6d59c2356,How does has_nested_base_model work in Python?,def has_nested_base_model(cls: type[IsDataclass]) -> bool:\n for f in fields(cls):\n field_type = f.type\n # Resolve forward references and other annotations\n origin = get...,autogen,synthetic_positive_v2
1,doc_c73087bba37c14c1e003ab940014b8a6d59c2356,Explain the has_nested_base_model logic,def has_nested_base_model(cls: type[IsDataclass]) -> bool:\n for f in fields(cls):\n field_type = f.type\n # Resolve forward references and other annotations\n origin = get...,autogen,synthetic_positive_v2
2,doc_c73087bba37c14c1e003ab940014b8a6d59c2356,Example usage of has_nested_base_model,def has_nested_base_model(cls: type[IsDataclass]) -> bool:\n for f in fields(cls):\n field_type = f.type\n # Resolve forward references and other annotations\n origin = get...,autogen,synthetic_positive_v2
3,doc_c73087bba37c14c1e003ab940014b8a6d59c2356,Best practices for has_nested_base_model,def has_nested_base_model(cls: type[IsDataclass]) -> bool:\n for f in fields(cls):\n field_type = f.type\n # Resolve forward references and other annotations\n origin = get...,autogen,synthetic_positive_v2
4,doc_c73087bba37c14c1e003ab940014b8a6d59c2356,How to implement has_nested_base_model?,def has_nested_base_model(cls: type[IsDataclass]) -> bool:\n for f in fields(cls):\n field_type = f.type\n # Resolve forward references and other annotations\n origin = get...,autogen,synthetic_positive_v2
5,doc_492e3d54a077116002387cd2ee7de8581e8c82d7,Explain the async create_mcp_websocket_connection logic,"async def create_mcp_websocket_connection(request: CreateWebSocketConnectionRequest):\n """"""Create WebSocket connection URL""""""\n try:\n session_id = str(uuid.uuid4())\n\n server...",autogen,synthetic_positive_v2
6,doc_492e3d54a077116002387cd2ee7de8581e8c82d7,Best practices for async create_mcp_websocket_connection,"async def create_mcp_websocket_connection(request: CreateWebSocketConnectionRequest):\n """"""Create WebSocket connection URL""""""\n try:\n session_id = str(uuid.uuid4())\n\n server...",autogen,synthetic_positive_v2
7,doc_492e3d54a077116002387cd2ee7de8581e8c82d7,How to implement async create_mcp_websocket_connection?,"async def create_mcp_websocket_connection(request: CreateWebSocketConnectionRequest):\n """"""Create WebSocket connection URL""""""\n try:\n session_id = str(uuid.uuid4())\n\n server...",autogen,synthetic_positive_v2
8,doc_492e3d54a077116002387cd2ee7de8581e8c82d7,Example usage of async create_mcp_websocket_connection,"async def create_mcp_websocket_connection(request: CreateWebSocketConnectionRequest):\n """"""Create WebSocket connection URL""""""\n try:\n session_id = str(uuid.uuid4())\n\n server...",autogen,synthetic_positive_v2
9,doc_492e3d54a077116002387cd2ee7de8581e8c82d7,How does async create_mcp_websocket_connection work in Python?,"async def create_mcp_websocket_connection(request: CreateWebSocketConnectionRequest):\n """"""Create WebSocket connection URL""""""\n try:\n session_id = str(uuid.uuid4())\n\n server...",autogen,synthetic_positive_v2


In [50]:
## drop column
df.drop(['framework','source'], axis=1, inplace=True)

In [51]:
## replace document_id with docid and numeric number for docid instead of string 
df.rename(columns = {'document_id':'docid'}, inplace = True)
df.head(10)

,docid,anchor,positive
0,doc_c73087bba37c14c1e003ab940014b8a6d59c2356,How does has_nested_base_model work in Python?,def has_nested_base_model(cls: type[IsDataclass]) -> bool:\n for f in fields(cls):\n field_type = f.type\n # Resolve forward references and other annotations\n origin = get...
1,doc_c73087bba37c14c1e003ab940014b8a6d59c2356,Explain the has_nested_base_model logic,def has_nested_base_model(cls: type[IsDataclass]) -> bool:\n for f in fields(cls):\n field_type = f.type\n # Resolve forward references and other annotations\n origin = get...
2,doc_c73087bba37c14c1e003ab940014b8a6d59c2356,Example usage of has_nested_base_model,def has_nested_base_model(cls: type[IsDataclass]) -> bool:\n for f in fields(cls):\n field_type = f.type\n # Resolve forward references and other annotations\n origin = get...
3,doc_c73087bba37c14c1e003ab940014b8a6d59c2356,Best practices for has_nested_base_model,def has_nested_base_model(cls: type[IsDataclass]) -> bool:\n for f in fields(cls):\n field_type = f.type\n # Resolve forward references and other annotations\n origin = get...
4,doc_c73087bba37c14c1e003ab940014b8a6d59c2356,How to implement has_nested_base_model?,def has_nested_base_model(cls: type[IsDataclass]) -> bool:\n for f in fields(cls):\n field_type = f.type\n # Resolve forward references and other annotations\n origin = get...
5,doc_492e3d54a077116002387cd2ee7de8581e8c82d7,Explain the async create_mcp_websocket_connection logic,"async def create_mcp_websocket_connection(request: CreateWebSocketConnectionRequest):\n """"""Create WebSocket connection URL""""""\n try:\n session_id = str(uuid.uuid4())\n\n server..."
6,doc_492e3d54a077116002387cd2ee7de8581e8c82d7,Best practices for async create_mcp_websocket_connection,"async def create_mcp_websocket_connection(request: CreateWebSocketConnectionRequest):\n """"""Create WebSocket connection URL""""""\n try:\n session_id = str(uuid.uuid4())\n\n server..."
7,doc_492e3d54a077116002387cd2ee7de8581e8c82d7,How to implement async create_mcp_websocket_connection?,"async def create_mcp_websocket_connection(request: CreateWebSocketConnectionRequest):\n """"""Create WebSocket connection URL""""""\n try:\n session_id = str(uuid.uuid4())\n\n server..."
8,doc_492e3d54a077116002387cd2ee7de8581e8c82d7,Example usage of async create_mcp_websocket_connection,"async def create_mcp_websocket_connection(request: CreateWebSocketConnectionRequest):\n """"""Create WebSocket connection URL""""""\n try:\n session_id = str(uuid.uuid4())\n\n server..."
9,doc_492e3d54a077116002387cd2ee7de8581e8c82d7,How does async create_mcp_websocket_connection work in Python?,"async def create_mcp_websocket_connection(request: CreateWebSocketConnectionRequest):\n """"""Create WebSocket connection URL""""""\n try:\n session_id = str(uuid.uuid4())\n\n server..."


# Generating Corpus

Main corpus is usually consists of unique query and answers 

In [52]:
# Remove duplicates based on docid (keeps first occurrence)
df_unique = df.drop_duplicates(subset=['docid'], keep='first')

In [53]:
# Reset docid to numeric starting from 0
df_unique['docid'] = range(len(df_unique))

/tmp/ipykernel_54/555291703.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_unique['docid'] = range(len(df_unique))


In [54]:
# Reset index if needed
df_unique = df_unique.reset_index(drop=True)
df_unique

,docid,anchor,positive
0,0,How does has_nested_base_model work in Python?,def has_nested_base_model(cls: type[IsDataclass]) -> bool:\n for f in fields(cls):\n field_type = f.type\n # Resolve forward references and other annotations\n origin = get...
1,1,Explain the async create_mcp_websocket_connection logic,"async def create_mcp_websocket_connection(request: CreateWebSocketConnectionRequest):\n """"""Create WebSocket connection URL""""""\n try:\n session_id = str(uuid.uuid4())\n\n server..."
2,2,Example usage of test_stop_bridge,"def test_stop_bridge(self, bridge):\n """"""Test stopping the bridge""""""\n assert bridge._running is True\n \n bridge.stop()\n \n assert bridge._running is False"
3,3,How does _put work in Python?,"def _put(self, item: T) -> None:\n self._queue.append(item)"
4,4,How to implement DataclassNestedUnionSyntaxNewMessage:\n message: str | int?,class DataclassNestedUnionSyntaxNewMessage:\n message: str | int
...,...,...,...
995,995,How does test_normalize_shell_output_handles_timeout work in Python?,"def test_normalize_shell_output_handles_timeout() -> None:\n entry = {\n ""stdout"": """",\n ""stderr"": """",\n ""outcome"": {""type"": ""timeout""},\n ""provider_data"": {""truncat..."
996,996,Explain the greet logic,"def greet(name: str) -> str:\n return f""Hello, {name}!"""
997,997,How to implement __init__?,def __init__(self):\n self.responses = DummyResponses()
998,998,Best practices for async _get_screenshot_async,"async def _get_screenshot_async(\n cls,\n computer: AsyncComputer,\n tool_call: ResponseComputerToolCall,\n ) -> str:\n action = tool_call.action\n if isinsta..."


In [55]:
## create a corpus with docid and positive(rename to text)
corpus=df_unique[['docid','positive']]
## rename positive to text and docid to docno
corpus.rename(columns = {'positive':'text','docid':'docno'}, inplace = True)
corpus

/tmp/ipykernel_54/4063860936.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  corpus.rename(columns = {'positive':'text','docid':'docno'}, inplace = True)


,docno,text
0,0,def has_nested_base_model(cls: type[IsDataclass]) -> bool:\n for f in fields(cls):\n field_type = f.type\n # Resolve forward references and other annotations\n origin = get...
1,1,"async def create_mcp_websocket_connection(request: CreateWebSocketConnectionRequest):\n """"""Create WebSocket connection URL""""""\n try:\n session_id = str(uuid.uuid4())\n\n server..."
2,2,"def test_stop_bridge(self, bridge):\n """"""Test stopping the bridge""""""\n assert bridge._running is True\n \n bridge.stop()\n \n assert bridge._running is False"
3,3,"def _put(self, item: T) -> None:\n self._queue.append(item)"
4,4,class DataclassNestedUnionSyntaxNewMessage:\n message: str | int
...,...,...
995,995,"def test_normalize_shell_output_handles_timeout() -> None:\n entry = {\n ""stdout"": """",\n ""stderr"": """",\n ""outcome"": {""type"": ""timeout""},\n ""provider_data"": {""truncat..."
996,996,"def greet(name: str) -> str:\n return f""Hello, {name}!"""
997,997,def __init__(self):\n self.responses = DummyResponses()
998,998,"async def _get_screenshot_async(\n cls,\n computer: AsyncComputer,\n tool_call: ResponseComputerToolCall,\n ) -> str:\n action = tool_call.action\n if isinsta..."


our corpus is ready

In [56]:
## make it a tsv file but remove extra index
corpus.to_csv('/kaggle/working/corpus.tsv',sep='\t',index=False)

# Generating Topics

In [57]:
# Get unique anchors (queries)
unique_queries = df['anchor'].unique()

In [58]:
# Randomly sample 69 queries
import random
random.seed(42)  # For reproducibility
sampled_queries = random.sample(list(unique_queries), 21)

In [59]:
# Create topics dataframe
topics_df = pd.DataFrame({
    'qid': [f'q{i+1}' for i in range(21)],
    'query': sampled_queries
})

In [60]:
topics_df

,qid,query
0,q1,Example usage of LangGraphAgentAdapter
1,q2,Explain the capabilities logic
2,q3,Example usage of serve
3,q4,Best practices for ChatGroq
4,q5,"How does TestResponseRewriting:\n """"""Test response content rewriting with wrap_model_call.""""""\n\n test_uppercase_response work in Python?"
5,q6,Example usage of truncate_messages
6,q7,Explain the async test_cache_streaming_list_without_create_result logic
7,q8,How does async test_selector_group_chat work in Python?
8,q9,Example usage of async test_pretty_run_result_streaming
9,q10,Explain the StreamArgs logic


In [61]:
# Save to TSV
topics_df.to_csv('/kaggle/working/topics.tsv', sep='\t', index=False)

In [62]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

In [63]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

HF_TOKEN=user_secrets.get_secret("HF_TOKEN")

In [64]:
from huggingface_hub import login
import os


login(token=HF_TOKEN)


- we will try to do it with llama-3.1-8b model first then will do it with llama-3.1-70b model

# Phase 2 — Depth-100 Pooling + LLM Judging
BM25 + Jina + CodeBERT each retrieve top-100 → union → judge only the pooled pairs.
This is standard TREC pooling methodology.

In [67]:
# Before indexing, force string type
corpus['docno'] = corpus['docno'].astype(str)

/tmp/ipykernel_54/3188760592.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  corpus['docno'] = corpus['docno'].astype(str)


In [98]:
# BM25 Index
BM25_INDEX_PATH = '/kaggle/working/custom_bm25_index'
indexer   = pt.IterDictIndexer(BM25_INDEX_PATH, overwrite=True)
index_ref = indexer.index(corpus.to_dict('records'))
bm25 = pt.BatchRetrieve(index_ref, wmodel='BM25', num_results=100)
print('BM25 index built.')
bm25(topics_df.head(2)).head(3)

BM25 index built.


/tmp/ipykernel_54/3352110468.py:5: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25 = pt.BatchRetrieve(index_ref, wmodel='BM25', num_results=100)


,qid,docid,docno,rank,score,query
0,q1,633,633,0,9.737878,Example usage of LangGraphAgentAdapter
1,q1,394,394,1,9.377459,Example usage of LangGraphAgentAdapter
2,q1,700,700,2,9.357109,Example usage of LangGraphAgentAdapter


In [99]:
# ── GraphCodeBERT dense index (Jina replacement) ──
graphcb       = pyterrier_dr.SBertBiEncoder('microsoft/graphcodebert-base', batch_size=64)
graphcb_index = pyterrier_dr.FlexIndex('/kaggle/working/custom_graphcb.flex')
(graphcb >> graphcb_index).index(corpus.to_dict('records'))
graphcb_retr  = graphcb >> graphcb_index.np_retriever()
print('GraphCodeBERT index built.')

# ── free before next model ──
gc.collect()
for i in range(torch.cuda.device_count()):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()


Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
indexing: 1000dvec [00:23, 42.97dvec/s]


GraphCodeBERT index built.


In [102]:
# CodeBERT dense index
codebert       = pyterrier_dr.SBertBiEncoder('microsoft/codebert-base')
codebert_index = pyterrier_dr.FlexIndex('/kaggle/working/custom_codebert.flex')
(codebert >> codebert_index).index(corpus.to_dict('records'))
codebert_retr  = codebert >> codebert_index.np_retriever()
print('CodeBERT index built.')

gc.collect()
for i in range(torch.cuda.device_count()):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()

indexing: 1000dvec [00:20, 48.19dvec/s]


CodeBERT index built.


In [104]:
# ── Depth-100 Pool ──
POOL_DEPTH = 100
bm25 = pt.BatchRetrieve(index_ref, wmodel='BM25', num_results=100)
graphcb_retr  = graphcb >> graphcb_index.np_retriever()
codebert_retr  = codebert >> codebert_index.np_retriever()

print('Running BM25...')
bm25_res = bm25(topics_df).query('rank < @POOL_DEPTH')
print('Running graphcodebert...')
graphcb_res = graphcb_retr(topics_df).query('rank < @POOL_DEPTH')
print('Running CodeBERT...')
cb_res   = codebert_retr(topics_df).query('rank < @POOL_DEPTH')

pool = (
    pd.concat([bm25_res, graphcb_res, cb_res], ignore_index=True)
    .drop_duplicates(subset=['qid', 'docno'])
    [['qid', 'docno']]
    .reset_index(drop=True)
)

# Attach code text + query text
pool = pool.merge(corpus.rename(columns={'text': 'code'}), on='docno', how='left')
pool = pool.merge(topics_df, on='qid', how='left')
pool = pool.dropna(subset=['code', 'query'])

print(f'Pool: {len(pool)} unique (query, doc) pairs')
print(f'Avg per query: {len(pool)/len(topics_df):.1f}')
pool.to_csv('/kaggle/working/pool.csv', index=False)

/tmp/ipykernel_54/486042576.py:3: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25 = pt.BatchRetrieve(index_ref, wmodel='BM25', num_results=100)


Running BM25...
Running graphcodebert...


NumpyRetriever scoring: 100%|██████████| 1/1 [00:00<00:00, 247.44docbatch/s]


Running CodeBERT...


NumpyRetriever scoring: 100%|██████████| 1/1 [00:00<00:00, 644.78docbatch/s]


Pool: 4871 unique (query, doc) pairs
Avg per query: 232.0


In [105]:
# Judge pooled pairs with best judge from Phase 0
# (If kernel restarted, set BEST_JUDGE_HF and BEST_JUDGE_4BIT manually)
# BEST_JUDGE_HF   = 'meta-llama/Llama-3.1-70B-Instruct'
# BEST_JUDGE_4BIT = True

pool = pd.read_csv('/kaggle/working/pool.csv')
judge_tok, judge_mdl = load_llm('Qwen/Qwen2.5-Coder-7B-Instruct')

pool_judged = []
CHECKPOINT_EVERY = 3

for i, (qid, group) in enumerate(tqdm(pool.groupby('qid'), desc='Judging queries')):
    scores = judge_batch(
        group['query'].tolist(), group['code'].tolist(),
        judge_tok, judge_mdl, batch_size=4
    )
    for (_, row), sc in zip(group.iterrows(), scores):
        pool_judged.append({'qid': row['qid'], 'docno': row['docno'], 'relevance': sc})

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(pool_judged).to_csv('/kaggle/working/pool_judged_ckpt.csv', index=False)
        print(f'  checkpoint @ query {i+1}')

pool_judged_df = pd.DataFrame(pool_judged)
pool_judged_df.to_csv('/kaggle/working/pool_judged.csv', index=False)
print(f'Judged {len(pool_judged_df)} pairs')
print(pool_judged_df['relevance'].value_counts().sort_index())
judge_mdl = None
judge_tok = None
gc.collect()
for i in range(torch.cuda.device_count()):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
for i in range(torch.cuda.device_count()):
    print(f"GPU {i} → {torch.cuda.memory_allocated(i)/1e9:.2f} GB")

Loading Qwen/Qwen2.5-Coder-7B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

  Loaded.


Judging queries:  14%|█▍        | 3/21 [02:46<15:51, 52.87s/it]

  checkpoint @ query 3


...

  checkpoint @ query 15



Judging queries:  86%|████████▌ | 18/21 [22:13<03:54, 78.05s/it]

  checkpoint @ query 18



Judging queries: 100%|██████████| 21/21 [26:42<00:00, 76.29s/it]


  checkpoint @ query 21
Judged 4871 pairs
relevance
0    4805
2      49
3      17
Name: count, dtype: int64
GPU 0 → 1.01 GB
GPU 1 → 0.01 GB


In [106]:
# Build TREC-format qrels
pool_judged_df = pd.read_csv('/kaggle/working/pool_judged.csv')
pool_judged_df['qid']   = pool_judged_df['qid'].astype(str)
pool_judged_df['docno'] = pool_judged_df['docno'].astype(str)

# TREC format (no header)
trec_qrels = pool_judged_df.copy()
trec_qrels.insert(1, 'Q0', '0')
trec_qrels[['qid','Q0','docno','relevance']].to_csv(
    '/kaggle/working/custom_qrels.tsv', sep='\t', index=False, header=False
)
# PyTerrier format (with header)
pool_judged_df[['qid','docno','relevance']].to_csv(
    '/kaggle/working/custom_qrels_pt.tsv', sep='\t', index=False
)
print(f'QRELs saved: {len(pool_judged_df)} judgements')
print(f'Relevant (rel>=2): {(pool_judged_df["relevance"]>=2).sum()}')

QRELs saved: 4871 judgements
Relevant (rel>=2): 66


# Phase 3 — Full Retrieval Benchmark
**Models**: BM25, Jina-Code, CodeBERT, MRL-LangChain, MRL-Dense, TctColBERT  
**Metrics**: nDCG@10, nDCG@5, MAP(rel=2), Recall@10

In [108]:
# Reload data (safe for fresh kernel)
import pyterrier as pt
if not pt.started(): pt.init()
import pyterrier_dr

corpus    = pd.read_csv('/kaggle/working/corpus.tsv', sep='\t')
topics_df = pd.read_csv('/kaggle/working/topics.tsv', sep='\t')
qrels     = pd.read_csv('/kaggle/working/custom_qrels_pt.tsv', sep='\t')

corpus['docno']    = corpus['docno'].astype(str)
topics_df['qid']   = topics_df['qid'].astype(str)
qrels['qid']       = qrels['qid'].astype(str)
qrels['docno']     = qrels['docno'].astype(str)

print(f'Corpus: {len(corpus)}, Topics: {len(topics_df)}, Qrels: {len(qrels)}')
print(qrels['relevance'].value_counts().sort_index())

Corpus: 1000, Topics: 21, Qrels: 4871
relevance
0    4805
2      49
3      17
Name: count, dtype: int64


/tmp/ipykernel_54/1072864294.py:3: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started(): pt.init()


In [111]:
# Rebuild all retrieval pipelines
from pyterrier.measures import nDCG, MAP, Recall

# BM25 — load from existing index (no re-indexing needed)
index_ref = pt.IndexRef.of('/kaggle/working/custom_bm25_index/data.properties')
bm25 = pt.BatchRetrieve(index_ref, wmodel='BM25', num_results=100)

# GraphCodeBERT — index ALREADY BUILT, just load retriever
graphcb       = pyterrier_dr.SBertBiEncoder('microsoft/graphcodebert-base', batch_size=64)
graphcb_index = pyterrier_dr.FlexIndex('/kaggle/working/custom_graphcb.flex')
# NO re-indexing — index is already on disk ✅
graphcb_retr  = graphcb >> graphcb_index.np_retriever()

# CodeBERT (base) — check if index exists first
codebert        = pyterrier_dr.SBertBiEncoder('microsoft/codebert-base')
codebert_index  = pyterrier_dr.FlexIndex('/kaggle/working/custom_codebert.flex')
if not codebert_index.built():
    (codebert >> codebert_index).index(corpus.to_dict('records'))
codebert_retr   = codebert >> codebert_index.np_retriever()

# MRL fine-tuned LangChain
mrl             = pyterrier_dr.SBertBiEncoder('shubharuidas/codebert-base-code-embed-mrl-langchain-langgraph')
mrl_index       = pyterrier_dr.FlexIndex('/kaggle/working/custom_mrl.flex')
if not mrl_index.built():
    (mrl >> mrl_index).index(corpus.to_dict('records'))
mrl_retr        = mrl >> mrl_index.np_retriever()

# MRL Dense Retriever
mrl2            = pyterrier_dr.SBertBiEncoder('shubharuidas/codebert-embed-base-dense-retriever')
mrl2_index      = pyterrier_dr.FlexIndex('/kaggle/working/custom_mrl2.flex')
if not mrl2_index.built():
    (mrl2 >> mrl2_index).index(corpus.to_dict('records'))
mrl2_retr       = mrl2 >> mrl2_index.np_retriever()

# TctColBERT
tct             = pyterrier_dr.TctColBert()
tct_index       = pyterrier_dr.FlexIndex('/kaggle/working/custom_tct.flex')
if not tct_index.built():
    (tct >> tct_index).index(corpus.to_dict('records'))
tct_retr        = tct >> tct_index.np_retriever()

print('All pipelines ready.')

/tmp/ipykernel_54/2089326234.py:6: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25 = pt.BatchRetrieve(index_ref, wmodel='BM25', num_results=100)
Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/283 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/441M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]


indexing: 0dvec [00:00, ?dvec/s]
indexing: 1dvec [00:03,  3.64s/dvec]
indexing: 101dvec [00:07, 16.54dvec/s]
indexing: 201dvec [00:10, 21.41dvec/s]
indexing: 301dvec [00:14, 23.62dvec/s]
indexing: 401dvec [00:18, 24.50dvec/s]
indexing: 501dvec [00:22, 24.96dvec/s]
indexing: 601dvec [00:26, 25.09dvec/s]
indexing: 701dvec [00:30, 25.26dvec/s]
indexing: 801dvec [00:34, 25.40dvec/s]
indexing: 1000dvec [00:37, 26.50dvec/s][A

All pipelines ready.


In [113]:
# ── Combined pt.Experiment with stat-sig vs BM25 baseline ──
METRICS = [nDCG@10, nDCG@5, MAP(rel=2), Recall@10]

results = pt.Experiment(
    [bm25, graphcb_retr, codebert_retr, mrl_retr, mrl2_retr, tct_retr],
    topics_df,
    qrels,
    eval_metrics=METRICS,
    names=['BM25', 'GraphcodeBert', 'CodeBERT-base', 'MRL-LangChain', 'MRL-Dense', 'TctColBERT'],
    baseline=0,           # BM25 as baseline → adds p-value + delta columns
    filter_by_qrels=True, # TREC standard: unjudged docs treated as non-relevant
    verbose=True,
)

results.to_csv('/kaggle/working/benchmark_results.csv', index=False)
print('\n=== BENCHMARK RESULTS ===')
display(results)

pt.Experiment: 100%|██████████| 6/6 [00:01<00:00,  4.97system/s]


=== BENCHMARK RESULTS ===


,name,R@10,nDCG@5,nDCG@10,AP(rel=2),R@10 +,R@10 -,R@10 p-value,nDCG@5 +,nDCG@5 -,nDCG@5 p-value,nDCG@10 +,nDCG@10 -,nDCG@10 p-value,AP(rel=2) +,AP(rel=2) -,AP(rel=2) p-value
0,BM25,0.489466,0.346191,0.394799,0.325243,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GraphcodeBert,0.096681,0.063770,0.064827,0.049686,1.0,14.0,0.002063,1.0,11.0,0.004454,1.0,14.0,0.000982,3.0,16.0,0.002944
2,CodeBERT-base,0.000000,0.000000,0.000000,0.005600,0.0,15.0,0.000025,0.0,12.0,0.000541,0.0,15.0,0.000091,3.0,16.0,0.000629
3,MRL-LangChain,0.526840,0.589829,0.584159,0.531909,4.0,8.0,0.745235,9.0,3.0,0.019652,9.0,7.0,0.087661,8.0,9.0,0.061966
4,MRL-Dense,0.526840,0.572254,0.566584,0.510433,4.0,8.0,0.745235,9.0,3.0,0.027869,9.0,7.0,0.115238,8.0,9.0,0.082957
5,TctColBERT,0.512554,0.472076,0.483066,0.428590,4.0,6.0,0.836681,7.0,5.0,0.196313,6.0,10.0,0.390620,6.0,12.0,0.300939


In [115]:
# Leaderboard Table
results_df = pd.read_csv('/kaggle/working/benchmark_results.csv')
num_cols = results_df.select_dtypes(include='number').columns
results_df[num_cols] = results_df[num_cols].round(4)
results_df = results_df.sort_values('nDCG@10', ascending=False).reset_index(drop=True)
results_df.insert(0, 'Rank', range(1, len(results_df)+1))

print('=== CODE RETRIEVAL BENCHMARK — Custom GenAI Corpus ===')
print(f'Queries: {len(topics_df)} | Corpus: {len(corpus)} docs')
print(f'QRELs: {len(qrels)}')
display(results_df[['Rank','name','nDCG@10','nDCG@5','AP(rel=2)','R@10']])

=== CODE RETRIEVAL BENCHMARK — Custom GenAI Corpus ===
Queries: 21 | Corpus: 1000 docs
QRELs: 4871


,Rank,name,nDCG@10,nDCG@5,AP(rel=2),R@10
0,1,MRL-LangChain,0.5842,0.5898,0.5319,0.5268
1,2,MRL-Dense,0.5666,0.5723,0.5104,0.5268
2,3,TctColBERT,0.4831,0.4721,0.4286,0.5126
3,4,BM25,0.3948,0.3462,0.3252,0.4895
4,5,GraphcodeBert,0.0648,0.0638,0.0497,0.0967
5,6,CodeBERT-base,0.0000,0.0000,0.0056,0.0000


In [117]:
# ── RQ Answer Summary ──
print('=' * 60)
r = results_df.set_index('name')

print('RQ1 — MRL vs Base CodeBERT')
base  = r.loc['CodeBERT-base','nDCG@10']
mrl_s = r.loc['MRL-LangChain','nDCG@10']
print(f'  CodeBERT-base  nDCG@10: {base:.4f}')
print(f'  MRL-LangChain  nDCG@10: {mrl_s:.4f}')
print(f'  Relative gain: {(mrl_s-base)/(base+1e-9)*100:+.1f}%')

print('\nRQ2 — Late Interaction (TctColBERT) vs best Bi-Encoder')
tct_s   = r.loc['TctColBERT','nDCG@10']
bi_best = results_df[results_df['name']!='TctColBERT']['nDCG@10'].max()
bi_name = results_df.loc[results_df['nDCG@10']==bi_best,'name'].values[0]
print(f'  TctColBERT    nDCG@10: {tct_s:.4f}')
print(f'  Best bi-enc ({bi_name}): {bi_best:.4f}')
print(f'  Gap: {tct_s-bi_best:+.4f}')


RQ1 — MRL vs Base CodeBERT
  CodeBERT-base  nDCG@10: 0.0000
  MRL-LangChain  nDCG@10: 0.5842
  Relative gain: +58420000000.0%

RQ2 — Late Interaction (TctColBERT) vs best Bi-Encoder
  TctColBERT    nDCG@10: 0.4831
  Best bi-enc (MRL-LangChain): 0.5842
  Gap: -0.1011


# RQ1 — MRL Embedding Dimension Sweep
Compares MRL-finetuned CodeBERT at dims 64/128/256/512/768 vs full-dim baseline.
Answers: *'Does MRL allow dimension reduction with minimal performance loss?'*

In [119]:
# ── RQ1 Setup: MRL Dimension Sweep ──
import pyterrier as pt

import pyterrier_dr
from pyterrier.measures import nDCG, MAP, Recall
import pandas as pd, numpy as np, gc, torch
from sentence_transformers import SentenceTransformer

# Load corpus and qrels (must have run Phase 1 & 2 first)
corpus    = pd.read_csv('/kaggle/working/corpus.tsv', sep='\t').astype(str)
topics_df = pd.read_csv('/kaggle/working/topics.tsv',           sep='\t').astype(str)
qrels_df  = pd.read_csv('/kaggle/working/custom_qrels_pt.tsv',  sep='\t').astype(str)
qrels_df['relevance'] = qrels_df['relevance'].astype(int)

MRL_MODEL = 'shubharuidas/codebert-base-code-embed-mrl-langchain-langgraph'
DIMS      = [64, 128, 256, 512, 768]   # 768 = full dim (baseline)
METRICS   = [nDCG @ 10, nDCG @ 5, MAP(rel=2), Recall @ 10]

print(f'Corpus: {len(corpus)} docs | Topics: {len(topics_df)} | Qrels: {len(qrels_df)}')


Corpus: 1000 docs | Topics: 21 | Qrels: 4871


In [124]:
from sentence_transformers import SentenceTransformer
import pyterrier as pt
import numpy as np

# ── Custom MRL Encoder (no internal attribute dependencies) ──
class MRLEncoder(pt.Transformer):
    """Wraps SentenceTransformer with truncate_dim for MRL inference."""
    def __init__(self, model_name, dim, batch_size=64):
        self.dim        = dim
        self.batch_size = batch_size
        self.model      = SentenceTransformer(model_name, truncate_dim=dim)
    
    def transform(self, inp):
        if 'query' in inp.columns:
            vecs = self.model.encode(
                inp['query'].tolist(), 
                batch_size=self.batch_size,
                normalize_embeddings=True,
                show_progress_bar=False
            )
            return inp.assign(query_vec=list(vecs))
        elif 'text' in inp.columns:
            vecs = self.model.encode(
                inp['text'].tolist(),
                batch_size=self.batch_size,
                normalize_embeddings=True,
                show_progress_bar=True
            )
            return inp.assign(doc_vec=list(vecs))
        return inp

print('MRLEncoder class ready.')


MRLEncoder class ready.


In [125]:
# ── Build MRL pipeline for each dimension ──
def build_mrl_pipeline(dim, model_name=MRL_MODEL):
    index_path = f'/kaggle/working/mrl_dim{dim}.flex'
    index      = pyterrier_dr.FlexIndex(index_path)
    encoder    = MRLEncoder(model_name, dim=dim, batch_size=64)
    
    if not index.built():
        print(f'  Indexing at dim={dim}...')
        (encoder >> index).index(corpus.to_dict('records'))
        print(f'  Done.')
    else:
        print(f'  Index at dim={dim} already exists, skipping.')
    
    # free GPU memory from encoder after indexing (keep index on disk)
    torch.cuda.empty_cache()
    
    return encoder >> index.np_retriever(num_results=100)

print('Building MRL indices for all dimensions...')
mrl_pipelines = {}

for dim in DIMS:
    print(f'\n[dim={dim}]')
    mrl_pipelines[dim] = build_mrl_pipeline(dim)
    gc.collect()
    torch.cuda.empty_cache()

print('\nAll indices built.')


Building MRL indices for all dimensions...

[dim=64]


  Indexing at dim=64...


indexing: 0dvec [00:00, ?dvec/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

indexing: 1dvec [00:02,  2.24s/dvec]

...

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

indexing: 801dvec [00:22, 38.01dvec/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

indexing: 1000dvec [00:25, 39.43dvec/s]


  Done.

All indices built.


In [130]:
# ── Run RQ1 Experiment: all dims in single pt.Experiment ──
print('Running RQ1 dimension sweep experiment...')

experiment_pipelines = [mrl_pipelines[d] for d in DIMS]
pipeline_names       = [f'MRL-{d}d' for d in DIMS]

rq1_results = pt.Experiment(
    experiment_pipelines,
    topics_df,
    qrels_df,
    eval_metrics=METRICS,
    names=pipeline_names,
    baseline=0,          # MRL-64d as baseline for significance vs full-dim
    correction='bonferroni',
)

rq1_results.to_csv('/kaggle/working/rq1_mrl_dims.csv', index=False)
print(rq1_results.to_string(index=False))
rq1_results


Running RQ1 dimension sweep experiment...


NumpyRetriever scoring: 100%|██████████| 1/1 [00:00<00:00, 640.94docbatch/s]


    name    R@10   nDCG@5  nDCG@10  AP(rel=2)  R@10 +  R@10 -  R@10 p-value  R@10 reject  R@10 p-value corrected  nDCG@5 +  nDCG@5 -  nDCG@5 p-value  nDCG@5 reject  nDCG@5 p-value corrected  nDCG@10 +  nDCG@10 -  nDCG@10 p-value  nDCG@10 reject  nDCG@10 p-value corrected  AP(rel=2) +  AP(rel=2) -  AP(rel=2) p-value  AP(rel=2) reject  AP(rel=2) p-value corrected
 MRL-64d 0.52684 0.589829 0.584159   0.527920     NaN     NaN           NaN        False                     NaN       NaN       NaN             NaN          False                       NaN        NaN        NaN              NaN           False                        NaN          NaN          NaN                NaN             False                          NaN
MRL-128d 0.52684 0.589829 0.584159   0.532478     0.0     0.0           NaN        False                     NaN       0.0       0.0             NaN          False                       NaN        0.0        0.0              NaN           False                        NaN 

,name,R@10,nDCG@5,nDCG@10,AP(rel=2),R@10 +,R@10 -,R@10 p-value,R@10 reject,R@10 p-value corrected,...,nDCG@10 +,nDCG@10 -,nDCG@10 p-value,nDCG@10 reject,nDCG@10 p-value corrected,AP(rel=2) +,AP(rel=2) -,AP(rel=2) p-value,AP(rel=2) reject,AP(rel=2) p-value corrected
0,MRL-64d,0.52684,0.589829,0.584159,0.527920,NaN,NaN,NaN,False,NaN,...,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,False,NaN
1,MRL-128d,0.52684,0.589829,0.584159,0.532478,0.0,0.0,NaN,False,NaN,...,0.0,0.0,NaN,False,NaN,2.0,2.0,0.308441,False,1.0
2,MRL-256d,0.52684,0.589829,0.584159,0.529362,0.0,0.0,NaN,False,NaN,...,0.0,0.0,NaN,False,NaN,2.0,2.0,0.406815,False,1.0
3,MRL-512d,0.52684,0.589829,0.584159,0.529436,0.0,0.0,NaN,False,NaN,...,0.0,0.0,NaN,False,NaN,2.0,2.0,0.295388,False,1.0
4,MRL-768d,0.52684,0.589829,0.584159,0.528903,0.0,0.0,NaN,False,NaN,...,0.0,0.0,NaN,False,NaN,3.0,1.0,0.426352,False,1.0


In [131]:
# ── RQ1 Table 6: MRL Dimensionality vs Performance ──
rq1_df = pd.read_csv('/kaggle/working/rq1_mrl_dims.csv')

print("Actual columns:", rq1_df.columns.tolist())  # debug: verify names

# Add storage column (% of full 768-dim)
rq1_df['dim']     = DIMS
rq1_df['storage'] = [f'{int(d/768*100)}%' for d in DIMS]

# Mark best row
best_ndcg = rq1_df['nDCG@10'].max()
rq1_df['best'] = rq1_df['nDCG@10'].apply(lambda x: '**' if x == best_ndcg else '')

print('\n=== TABLE 6: MRL Dimension Sweep ===')
# ← use actual column names from your CSV
print(rq1_df[['name','dim','nDCG@10','nDCG@5','AP(rel=2)','R@10','storage']].to_string(index=False))

rq1_df.to_csv('/kaggle/working/rq1_table6.csv', index=False)


Actual columns: ['name', 'R@10', 'nDCG@5', 'nDCG@10', 'AP(rel=2)', 'R@10 +', 'R@10 -', 'R@10 p-value', 'R@10 reject', 'R@10 p-value corrected', 'nDCG@5 +', 'nDCG@5 -', 'nDCG@5 p-value', 'nDCG@5 reject', 'nDCG@5 p-value corrected', 'nDCG@10 +', 'nDCG@10 -', 'nDCG@10 p-value', 'nDCG@10 reject', 'nDCG@10 p-value corrected', 'AP(rel=2) +', 'AP(rel=2) -', 'AP(rel=2) p-value', 'AP(rel=2) reject', 'AP(rel=2) p-value corrected']

=== TABLE 6: MRL Dimension Sweep ===
    name  dim  nDCG@10   nDCG@5  AP(rel=2)    R@10 storage
 MRL-64d   64 0.584159 0.589829   0.527920 0.52684      8%
MRL-128d  128 0.584159 0.589829   0.532478 0.52684     16%
MRL-256d  256 0.584159 0.589829   0.529362 0.52684     33%
MRL-512d  512 0.584159 0.589829   0.529436 0.52684     66%
MRL-768d  768 0.584159 0.589829   0.528903 0.52684    100%


In [133]:
# ── RQ1 Figure 1: nDCG@10 vs Embedding Dimension ──
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

rq1_df = pd.read_csv('/kaggle/working/rq1_table6.csv')

# Also load full CodeBERT (non-MRL) for flat reference line
# If you have it from Phase 3, load it; else use a placeholder
try:
    benchmark = pd.read_csv('/kaggle/working/benchmark_results.csv')
    codebert_ndcg = float(benchmark[benchmark['name']=='CodeBERT-base']['nDCG@10'].values[0])
except:
    codebert_ndcg = None   # skip reference line if not available

fig, ax = plt.subplots(figsize=(6, 4))

# MRL curve
ax.plot(rq1_df['dim'], rq1_df['nDCG@10'],
        marker='o', linewidth=2, color='#2ecc71', label='MRL-finetuned CodeBERT')

# CodeBERT-base flat reference line
if codebert_ndcg:
    ax.axhline(codebert_ndcg, linestyle='--', color='#e74c3c', linewidth=1.5,
               label=f'CodeBERT-base (768d) = {codebert_ndcg:.3f}')

# 5% drop zone
full_dim_ndcg = float(rq1_df[rq1_df['dim']==768]['nDCG@10'].values[0])
ax.axhspan(full_dim_ndcg * 0.95, full_dim_ndcg, alpha=0.08, color='green',
           label='<5% drop zone')

ax.set_xlabel('Embedding Dimension', fontsize=12)
ax.set_ylabel('nDCG@10', fontsize=12)
ax.set_title('RQ1: MRL Embedding Dimension vs Retrieval Performance', fontsize=11)
ax.set_xticks(rq1_df['dim'])
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/rq1_figure1.pdf', bbox_inches='tight')
plt.savefig('/kaggle/working/rq1_figure1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to rq1_figure1.pdf and rq1_figure1.png')


Figure saved to rq1_figure1.pdf and rq1_figure1.png


In [132]:
# In the LaTeX table cell, change:
# MAP(rel=2) → AP(rel=2)
# Recall@10  → R@10

# In the RQ1 Answer cell, change:
rq1_df = pd.read_csv('/kaggle/working/rq1_table6.csv')
full   = rq1_df[rq1_df['dim']==768]['nDCG@10'].values[0]
best_reduced = rq1_df[rq1_df['dim']<768].sort_values('nDCG@10', ascending=False).iloc[0]

print(f"Full-dim (768d) nDCG@10  : {full:.4f}")
print(f"Best reduced dim         : {int(best_reduced['dim'])}d")
print(f"nDCG@10 at {int(best_reduced['dim'])}d          : {best_reduced['nDCG@10']:.4f}")
print(f"Performance retained     : {best_reduced['nDCG@10']/full*100:.1f}%")
print(f"Storage reduction        : {best_reduced['storage']}")

Full-dim (768d) nDCG@10  : 0.5842
Best reduced dim         : 64d
nDCG@10 at 64d          : 0.5842
Performance retained     : 100.0%
Storage reduction        : 8%


In [134]:
# ── RQ1 Answer ──
rq1_df = pd.read_csv('/kaggle/working/rq1_table6.csv')
full   = rq1_df[rq1_df['dim']==768]['nDCG@10'].values[0]
best_reduced = rq1_df[rq1_df['dim']<768].sort_values('nDCG@10', ascending=False).iloc[0]

print('='*55)
print('RQ1 ANSWER — MRL Embedding Efficiency')
print('='*55)
print(f"Full-dim (768d) nDCG@10  : {full:.4f}")
print(f"Best reduced dim         : {int(best_reduced['dim'])}d")
print(f"nDCG@10 at {int(best_reduced['dim'])}d          : {best_reduced['nDCG@10']:.4f}")
print(f"Performance retained     : {best_reduced['nDCG@10']/full*100:.1f}%")
print(f"Storage reduction        : {best_reduced['storage']}")
print()
print(f">> MRL retains {best_reduced['nDCG@10']/full*100:.1f}% of full-dim nDCG@10")
print(f"   at {int(best_reduced['dim'])}d ({best_reduced['storage']} storage), answering RQ1 affirmatively.")

RQ1 ANSWER — MRL Embedding Efficiency
Full-dim (768d) nDCG@10  : 0.5842
Best reduced dim         : 64d
nDCG@10 at 64d          : 0.5842
Performance retained     : 100.0%
Storage reduction        : 8%

>> MRL retains 100.0% of full-dim nDCG@10
   at 64d (8% storage), answering RQ1 affirmatively.


# Remaining Early works

In [21]:
model_name = "meta-llama/Llama-3.1-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token 
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
2026-03-28 18:40:24.480196: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774723224.877038      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774723224.978667      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774723225.863012      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774723225.863050      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774723225.863053      55

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [22]:
RELEVANCE_PROMPT = """You are an expert code relevance assessor for information retrieval research.

Task: Rate the relevance of a code snippet to a natural language query on a scale of 0-3.

Relevance Scale:
- 0: Not relevant - The code has no relation to the query
- 1: Marginally relevant - The code is tangentially related but does not answer the query
- 2: Relevant - The code partially addresses the query or provides related functionality
- 3: Highly relevant - The code directly and completely answers the query

Query: {query}

Code Snippet:
{code}

Instructions:
1. Analyze the semantic relationship between the query and code
2. Consider functionality, intent, and implementation details
3. Provide ONLY a single digit (0, 1, 2, or 3) as your response

Relevance Score:"""


In [23]:
def judge_relevance(queries, codes, batch_size=8):
    """Judge multiple query-code pairs in parallel"""
    
    results = []
    
    for i in range(0, len(queries), batch_size):
        batch_queries = queries[i:i+batch_size]
        batch_codes = codes[i:i+batch_size]
        
        prompts = [
            RELEVANCE_PROMPT.format(query=q, code=c[:1000])
            for q, c in zip(batch_queries, batch_codes)
        ]
        
        # Tokenize with padding
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048,
            return_attention_mask=True
        )
        
        # Move to device
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_new_tokens=5,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        
        # Decode
        for j, output in enumerate(outputs):
            input_length = inputs['input_ids'][j].shape[0]
            generated = output[input_length:]
            response = tokenizer.decode(generated, skip_special_tokens=True)
            
            try:
                score = int(response.strip()[0])
                if score not in [0, 1, 2, 3]:
                    score = 0
            except:
                score = 0
            
            results.append(score)
    
    return results


In [ ]:
qrels_8b = []
batch_size = 16  # Larger batch size for dual GPU

# Process each query
for idx, topic in tqdm(topics_df.iterrows(), total=len(topics_df), desc="Processing queries"):
    qid = topic['qid']
    query = topic['query']
    
    # Prepare batch: all documents for this query
    queries_batch = [query] * len(corpus)
    codes_batch = corpus['text'].tolist()
    
    # Judge all documents for this query in batches
    relevance_scores = judge_relevance(queries_batch, codes_batch, batch_size)
    
    # Store results
    for docno, relevance in zip(corpus['docno'], relevance_scores):
        qrels_8b.append({
            'qid': qid,
            'docno': docno,
            'relevance': relevance
        })
    
    # Save checkpoint every 10 queries
    if (idx + 1) % 10 == 0:
        checkpoint_df = pd.DataFrame(qrels_8b)
        checkpoint_df.to_csv(f'qrels_checkpoint_{idx+1}.csv', index=False)

print(f"\n Generated {len(qrels_8b)} relevance judgments")


Processing queries:  38%|███▊      | 8/21 [48:41<1:19:23, 366.44s/it]

In [24]:
qrels_df = pd.DataFrame(qrels_8b)

# TREC format: qid Q0 docno relevance
qrels_df['Q0'] = 0
qrels_df = qrels_df[['qid', 'Q0', 'docno', 'relevance']]
qrels_df

,qid,Q0,docno,relevance
0,q1,0,0,3
1,q1,0,1,2
2,q1,0,2,0
3,q1,0,3,0
4,q1,0,4,3
...,...,...,...,...
20995,q21,0,995,2
20996,q21,0,996,2
20997,q21,0,997,2
20998,q21,0,998,3


In [25]:
qrels_df.to_csv('/kaggle/working/qrels_8b.tsv', sep='\t', index=False, header=False)

- Now we will do it with 70b parameters

In [ ]:
model_name = "meta-llama/Llama-3.1-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token 
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

In [ ]:
RELEVANCE_PROMPT = """You are an expert code relevance assessor for information retrieval research.

Task: Rate the relevance of a code snippet to a natural language query on a scale of 0-3.

Relevance Scale:
- 0: Not relevant - The code has no relation to the query
- 1: Marginally relevant - The code is tangentially related but does not answer the query
- 2: Relevant - The code partially addresses the query or provides related functionality
- 3: Highly relevant - The code directly and completely answers the query

Query: {query}

Code Snippet:
{code}

Instructions:
1. Analyze the semantic relationship between the query and code
2. Consider functionality, intent, and implementation details
3. Provide ONLY a single digit (0, 1, 2, or 3) as your response

Relevance Score:"""

In [ ]:
qrels_70b = []
batch_size = 16  # Larger batch size for dual GPU

# Process each query
for idx, topic in tqdm(topics_df.iterrows(), total=len(topics_df), desc="Processing queries"):
    qid = topic['qid']
    query = topic['query']
    
    # Prepare batch: all documents for this query
    queries_batch = [query] * len(corpus)
    codes_batch = corpus['text'].tolist()
    
    # Judge all documents for this query in batches
    relevance_scores = judge_relevance(queries_batch, codes_batch, batch_size)
    
    # Store results
    for docno, relevance in zip(corpus['docno'], relevance_scores):
        qrels_70b.append({
            'qid': qid,
            'docno': docno,
            'relevance': relevance
        })
    
    # Save checkpoint every 10 queries
    if (idx + 1) % 10 == 0:
        checkpoint_df = pd.DataFrame(qrels_70b)
        checkpoint_df.to_csv(f'qrels_checkpoint_{idx+1}.csv', index=False)

print(f"\n Generated {len(qrels_70b)} relevance judgments")


In [ ]:
qrels_df_70b = pd.DataFrame(qrels_70b)

# TREC format: qid Q0 docno relevance
qrels_df_70b['Q0'] = 0
qrels_df_70b = qrels_df_70b[['qid', 'Q0', 'docno', 'relevance']]
qrels_df_70b

In [ ]:
qrels_df_70b.to_csv('/kaggle/working/qrels_70b.tsv', sep='\t', index=False, header=False)

# codeDL analysis using colBERT

In [26]:
%pip install -q faiss-cpu pyterrier-dr

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 77.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.3/208.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.0/149.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [27]:
import pyterrier_dr

model = pyterrier_dr.TctColBert()

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/441M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [39]:
index = pyterrier_dr.FlexIndex('/kaggle/working/codedl.tctcolbert.flex')

In [41]:
corpus["docno"] = corpus["docno"].astype(str)

/tmp/ipykernel_55/3750218120.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  corpus["docno"] = corpus["docno"].astype(str)


In [44]:
idx_pipeline = model >> index
idx_pipeline.index(corpus.to_dict('records') )

indexing: 1000dvec [00:31, 32.23dvec/s]


FlexIndex('/kaggle/working/codedl.tctcolbert.flex')

In [45]:
retr_pipeline = model >> index.np_retriever()

In [49]:
%pip install -q python-terrier

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [52]:
qrels_df

,qid,Q0,docno,relevance
0,q1,0,0,3
1,q1,0,1,2
2,q1,0,2,0
3,q1,0,3,0
4,q1,0,4,3
...,...,...,...,...
20995,q21,0,995,2
20996,q21,0,996,2
20997,q21,0,997,2
20998,q21,0,998,3


In [55]:
qrels_df["qid"] = qrels_df["qid"].astype(str)
qrels_df["docno"] = qrels_df["docno"].astype(str)
qrels_df['Q0']=qrels_df['Q0'].astype(str)


In [57]:
topics_df

,qid,query
0,q1,Example usage of LangGraphAgentAdapter
1,q2,Explain the capabilities logic
2,q3,Example usage of serve
3,q4,Best practices for ChatGroq
4,q5,"How does TestResponseRewriting:\n """"""Test r..."
5,q6,Example usage of truncate_messages
6,q7,Explain the async test_cache_streaming_list_wi...
7,q8,How does async test_selector_group_chat work i...
8,q9,Example usage of async test_pretty_run_result_...
9,q10,Explain the StreamArgs logic


In [59]:
topics_df['qid']=topics_df['qid'].astype(str)

In [61]:
import pyterrier as pt
from pyterrier.measures import nDCG, MAP


result = pt.Experiment(
    [retr_pipeline],
    topics_df,
    qrels_df,
    eval_metrics=[nDCG@10, MAP(rel=2)],
    names=["tctColBERT"],
    filter_by_qrels=False,
    verbose=False
)

result


NumpyRetriever scoring: 100%|██████████| 1/1 [00:00<00:00, 38.79docbatch/s]


,name,AP(rel=2),nDCG@10
0,tctColBERT,0.59472,0.617437


In [67]:
codebert = pyterrier_dr.SBertBiEncoder("microsoft/codebert-base")

In [70]:
codebert_index = pyterrier_dr.FlexIndex("/kaggle/working/codebert.flex")

indexing_pipeline = codebert >> codebert_index
indexing_pipeline.index(corpus.to_dict("records"))

indexing: 1000dvec [00:20, 48.22dvec/s]


FlexIndex('/kaggle/working/codebert.flex')

In [71]:
retr_pipeline = codebert >> index.np_retriever()

In [72]:
import pyterrier as pt
from pyterrier.measures import nDCG, MAP


result = pt.Experiment(
    [retr_pipeline],
    topics_df,
    qrels_df,
    eval_metrics=[nDCG@10, MAP(rel=2)],
    names=["codeBERT"],
    filter_by_qrels=False,
    verbose=False
)

result

NumpyRetriever scoring: 100%|██████████| 1/1 [00:00<00:00, 378.17docbatch/s]


,name,AP(rel=2),nDCG@10
0,codeBERT,0.523545,0.347075


In [73]:
finetuned_codebert = pyterrier_dr.SBertBiEncoder("shubharuidas/codebert-base-code-embed-mrl-langchain-langgraph")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/283 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

In [75]:
codebert_index = pyterrier_dr.FlexIndex("/kaggle/working/finetuned_codebert.flex")

indexing_pipeline = finetuned_codebert >> codebert_index
indexing_pipeline.index(corpus.to_dict("records"))

indexing: 1000dvec [00:20, 48.46dvec/s]


FlexIndex('/kaggle/working/finetuned_codebert.flex')

In [76]:
retr_pipeline = finetuned_codebert >> index.np_retriever()

In [78]:
import pyterrier as pt
from pyterrier.measures import nDCG, MAP


result = pt.Experiment(
    [retr_pipeline],
    topics_df,
    qrels_df,
    eval_metrics=[nDCG@10, MAP(rel=2)],
    names=["finetunedcodeBERT"],
    filter_by_qrels=False,
    verbose=False
)

result

NumpyRetriever scoring: 100%|██████████| 1/1 [00:00<00:00, 499.38docbatch/s]


,name,AP(rel=2),nDCG@10
0,finetunedcodeBERT,0.565216,0.482891


In [79]:
finetuned_codebert = pyterrier_dr.SBertBiEncoder("shubharuidas/codebert-embed-base-dense-retriever")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/283 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

In [80]:
codebert_index = pyterrier_dr.FlexIndex("/kaggle/working/finetuned_codebert2.flex")

indexing_pipeline = finetuned_codebert >> codebert_index
indexing_pipeline.index(corpus.to_dict("records"))

indexing: 1000dvec [00:20, 49.88dvec/s]


FlexIndex('/kaggle/working/finetuned_codebert2.flex')

In [81]:
retr_pipeline = finetuned_codebert >> index.np_retriever()

In [82]:
import pyterrier as pt
from pyterrier.measures import nDCG, MAP


result = pt.Experiment(
    [retr_pipeline],
    topics_df,
    qrels_df,
    eval_metrics=[nDCG@10, MAP(rel=2)],
    names=["finetunedcodeBERT"],
    filter_by_qrels=False,
    verbose=False
)

result

NumpyRetriever scoring: 100%|██████████| 1/1 [00:00<00:00, 1005.59docbatch/s]


,name,AP(rel=2),nDCG@10
0,finetunedcodeBERT,0.55262,0.465302
